# 1. Setup

## 1.1 Importing Packages

In [1]:
#%pip install xarray rioxarray pandas matplotlib rasterio
import geopandas as gpd
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import rioxarray
from rasterio import rio
from rasterio import features
from rasterio.enums import Resampling
import numpy as np
from rasterio.transform import Affine

## 1.2 Importing and masking S-2 data

In [14]:
def forestmasking(path, raw_mask):
    s2_data = rioxarray.open_rasterio(path)
    aligned_mask = raw_mask.rio.reproject_match(s2_data, resampling=Resampling.nearest)
    aligned_mask = aligned_mask.squeeze("band")
    return s2_data.where(aligned_mask == 1)

forestmask = rioxarray.open_rasterio(r"C:\Users\miles\OneDrive\Dokumente\ROOT\sentinel2\forestmask3035.tif")

n22 = forestmasking(r"C:\Users\miles\OneDrive\Dokumente\ROOT\sentinel2\s2_nord_22.tif", forestmask)
n23 = forestmasking(r"C:\Users\miles\OneDrive\Dokumente\ROOT\sentinel2\s2_nord_23.tif", forestmask)
s20 = forestmasking(r"C:\Users\miles\OneDrive\Dokumente\ROOT\sentinel2\s2_süd_20.tif", forestmask)
s22 = forestmasking(r"C:\Users\miles\OneDrive\Dokumente\ROOT\sentinel2\s2_süd_22.tif", forestmask)

## 1.3 Saving masked data



In [15]:
n22.attrs['long_name'] = [str(b) for b in n22.band.values]
n22.rio.to_raster(r"C:\Users\miles\OneDrive\Dokumente\ROOT\sentinel2\masked_s2\n22.tif")

n23.attrs['long_name'] = [str(b) for b in n23.band.values]
n23.rio.to_raster(r"C:\Users\miles\OneDrive\Dokumente\ROOT\sentinel2\masked_s2\n23.tif")

s20.attrs['long_name'] = [str(b) for b in s20.band.values]
s20.rio.to_raster(r"C:\Users\miles\OneDrive\Dokumente\ROOT\sentinel2\masked_s2\s20.tif")

s22.attrs['long_name'] = [str(b) for b in s22.band.values]
s22.rio.to_raster(r"C:\Users\miles\OneDrive\Dokumente\ROOT\sentinel2\masked_s2\s22.tif")

## 2.3 Rasterizing training data

The purpose of this function is to rasterize the training polygons. Because the polygons are often very small, I only want to use pixels as training data, which are covered by one class (undisturbed, deadwood, clear) by more than 80%. Otherwise the training data will be very messy and full of mixed pixels.


### 2.3.1 Defining function

The function does the following:

Masks the sent-2 data with the forestmask
Sets everything in AOI thats not already a polygon to 3 (undisturbed)
rasterizes the training polygons to 2m resolution
Sets every pixel to NA
For loop:
     &nbsp; &nbsp; &nbsp;  sets every 10-by-10 pixel with more than 80% (20 2-by-2 pixels) covered with a 1-polygon to 1,
 &nbsp; &nbsp; &nbsp;   sets every pixel with more than 80% covered with a 2-polygon to 2,
  &nbsp; &nbsp; &nbsp;sets every pixel with more than 80% covered with a 3-polygon to 3,

In [16]:
def rasterize_training(train_path, aoi_path, data_path):

    s2_data = forestmasking(data_path, forestmask)
    train_data = gpd.read_file(train_path)
    train_data["geometry"] = train_data.make_valid()
    aoi = gpd.read_file(aoi_path)
    aoi["geometry"] = aoi.make_valid()

    scale = 5

    # Einheitliche CRS
    train_data = train_data.to_crs(s2_data.rio.crs)
    aoi = aoi.to_crs(s2_data.rio.crs)

    # here, all the non-defined areas will be set to "3" (undisturbed)
    train_footprint = train_data.union_all()
    undist_geom = aoi.geometry.union_all().difference(train_footprint)

    undist_data = gpd.GeoDataFrame(geometry=[undist_geom], crs=aoi.crs)
    undist_data = undist_data.explode(index_parts=False).reset_index(drop=True)
    undist_data["class"] = 3 # alles was nicht schon definiert ist, ist undisturbed = 3

    undist_data = undist_data[~undist_data.geometry.is_empty]

    # Combine and ensure it remains a GeoDataFrame
    combined_df = pd.concat([train_data, undist_data], ignore_index=True)
    train_data = gpd.GeoDataFrame(combined_df, geometry='geometry', crs=aoi.crs)

    # um die mixed pixels rauszunehmen, rasterisiere ich die Polygone in 2m*2m Auflösung
    # alle polygon die nicht zu mindestens 80% rein sind werden dann zu NAs
    high_res_transform = s2_data.rio.transform() * Affine.scale(0.2, 0.2)

    # MODIFIED: Use int() to ensure dimensions are integers
    high_res_shape = (int(s2_data.rio.height * scale), int(s2_data.rio.width * scale))

    shapes = ((geom, value) for geom, value in zip(train_data.geometry, train_data["class"]))

    high_res_raster = features.rasterize(
        shapes=shapes,
        out_shape=high_res_shape,
        transform=high_res_transform,
        fill=0,
        dtype=np.int16
    )

    reshaped = high_res_raster.reshape(
        s2_data.rio.height, scale,
        s2_data.rio.width, scale
    )

    final_mask_2d = np.full((s2_data.rio.height, s2_data.rio.width), np.nan)

    for class_id in [1, 2, 3]:

        class_count = np.sum(reshaped == class_id, axis=(1, 3))
        final_mask_2d[class_count >= 20] = class_id

    new_band = s2_data.isel(band=[0]).copy(data=np.expand_dims(final_mask_2d, axis=0))
    new_band.name = "training_class"
    new_band.coords["band"] = ["training_class"]
    combined_data = new_band

    return combined_data

### 2.3.2 Apply Function

In [17]:
rast_train_n22 = rasterize_training(train_path=r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\other and original data\deadwood_samples_miles_nord_2022.gpkg", aoi_path=r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\AOIs\aoi_n22.geojson", data_path=r"C:\Users\miles\OneDrive\Dokumente\ROOT\sentinel2\s2_nord_22.tif")

rast_train_n23 = rasterize_training(train_path=r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\other and original data\deadwood_samples_miles_nord_2023.gpkg", aoi_path=r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\AOIs\aoi_n23.geojson", data_path=r"C:\Users\miles\OneDrive\Dokumente\ROOT\sentinel2\s2_nord_23.tif")

rast_train_s20 = rasterize_training(train_path=r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\other and original data\deadwood_samples_miles_sued_2020.gpkg", aoi_path=r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\AOIs\aoi_s20.geojson", data_path=r"C:\Users\miles\OneDrive\Dokumente\ROOT\sentinel2\s2_süd_20.tif")

rast_train_s22 = rasterize_training(train_path=r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\other and original data\deadwood_samples_miles_sued_2022.gpkg", aoi_path=r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\AOIs\aoi_s22.geojson", data_path=r"C:\Users\miles\OneDrive\Dokumente\ROOT\sentinel2\s2_süd_22.tif")

## 2.3.3 Save results


In [18]:
rast_train_n22.attrs['long_name'] = [str(b) for b in rast_train_n22.band.values]
rast_train_n22.rio.to_raster(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_clean\traindata_n22.tif")

rast_train_n23.attrs['long_name'] = [str(b) for b in rast_train_n23.band.values]
rast_train_n23.rio.to_raster(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_clean\traindata_n23.tif")

rast_train_s20.attrs['long_name'] = [str(b) for b in rast_train_s20.band.values]
rast_train_s20.rio.to_raster(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_clean\traindata_s20.tif")

rast_train_s22.attrs['long_name'] = [str(b) for b in rast_train_s22.band.values]
rast_train_s22.rio.to_raster(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_clean\traindata_s22.tif")

## 3. Stacking and Saving
Hier habe ich die saubere trainingsinfo auf die sentinel-2 bänder gelegt. Außerdem mache ich alle Pixel zu NA, die in der Klassifizierung oder der s2-daten NA sind.

In [24]:
bandnames = ["blue", "green", "red", "rededge1", "rededge2", "rededge3", "nir", "nir_narrow", "swir1", "swir2", "trainclass"]

full_n22 = xr.concat([n22, rast_train_n22], dim = "band")
mask = full_n22.notnull().all(dim="band")
full_n22 = full_n22.where(mask) #damit sind die trainingsdaten dann auch für den Wald maskiert
full_n22 = full_n22.assign_coords(band = bandnames)
full_n22.attrs['long_name'] = bandnames
full_n22 = full_n22 / 10000
full_n22.rio.to_raster(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_clean_full\n22.tif")


full_n23 = xr.concat([n23, rast_train_n23], dim="band")
mask_n23 = full_n23.notnull().all(dim="band")
full_n23 = full_n23.where(mask_n23)
full_n23 = full_n23.assign_coords(band=bandnames)
full_n23.attrs['long_name'] = bandnames
full_n23 = full_n23 / 10000
full_n23.rio.to_raster(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_clean_full\n23.tif")

# --- s20 ---
full_s20 = xr.concat([s20, rast_train_s20], dim="band")
mask_s20 = full_s20.notnull().all(dim="band")
full_s20 = full_s20.where(mask_s20)
full_s20 = full_s20.assign_coords(band=bandnames)
full_s20.attrs['long_name'] = bandnames
full_s20 = full_s20 / 10000
full_s20.rio.to_raster(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_clean_full\s20.tif")

# --- s22 ---
full_s22 = xr.concat([s22, rast_train_s22], dim="band")
mask_s22 = full_s22.notnull().all(dim="band")
full_s22 = full_s22.where(mask_s22)
full_s22 = full_s22.assign_coords(band=bandnames)
full_s22.attrs['long_name'] = bandnames
full_s22 = full_s22 / 10000
full_s22.rio.to_raster(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_clean_full\s22.tif")


In [22]:
print(f"Number of layers in stack: {full_n22.rio.count}")
print(f"Number of names provided: {len(bandnames)}")

Number of layers in stack: 11
Number of names provided: 11
